# DeepLure: color-invariant saree design recognition
This repository implements image-level identification and pair verification with a compact PyTorch embedding model. **AI tools used: Yes.** Read `AI_DISCLOSURE.md` and the execution report before presenting results.

Exact design means distinctive motif and spatial layout, including visible border/pallu. Category labels are not design labels. The executed experiment uses real public images for instance-contrastive training; its held-out recoloring benchmark is synthetic, not natural cross-colorway evidence.

This submission notebook has clean outputs. A separately executed smoke notebook is software evidence only. Real-data results from the coding-agent run are recorded in `reports/`. The first experiment was invalidated after duplicate-screening misses; the corrected experiment and prior test exposure are disclosed in `reports/CORRECTION_LOG.md`. This notebook requires the repository/ZIP contents and authorized local data; it does not download proprietary data or silently substitute fixtures.

**EXECUTED SOFTWARE SMOKE ONLY.** Generated geometric images; pretrained initialization. No natural saree recognition or independent validation is established by this notebook.

In [1]:
from pathlib import Path
import os, sys, subprocess, json, datetime
# Place this notebook inside the extracted repository's notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "saree").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Extract the full project ZIP; run from its root or notebooks directory.")
MODE = "synthetic_smoke"  # explicit alternatives: "real" or "synthetic_smoke"
DATA_ROOT = PROJECT_ROOT / "private/data"
OUTPUT_ROOT = PROJECT_ROOT / "private/notebook_smoke_v2"
DEVICE = "auto"  # cpu, cuda, mps, auto
TRAINING_MODE = "instance"  # supervised requires independently supported design/capture annotations
ANNOTATIONS = None  # optional private JSON mapping relative paths to evidenced annotations
CONFIG = PROJECT_ROOT / "configs/cpu.json"
INSTALL_DEPENDENCIES = False  # set True only if this kernel has not been bootstrapped
if MODE not in ("real", "synthetic_smoke"):
    raise ValueError("Explicit mode required")
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")])
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
os.environ["TORCH_HOME"] = str(PROJECT_ROOT / "private/torch-cache")
os.environ["MPLCONFIGDIR"] = str(PROJECT_ROOT / "private/matplotlib-cache")
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
if OUTPUT_ROOT.exists():
    raise FileExistsError("Use a new OUTPUT_ROOT to preserve frozen splits and one-use test results.")
OUTPUT_ROOT.mkdir(parents=True)
from saree.common import read_json, write_json, device_for, seed_all
seed_all(17)
device = device_for(DEVICE)
config = read_json(CONFIG)
print({"mode": MODE, "device": str(device), "training_mode": TRAINING_MODE})

{'mode': 'synthetic_smoke', 'device': 'cpu', 'training_mode': 'instance'}


## Audit and protocol
Data discovery and licensing are documented in `reports/AUDIT.md` and `reports/SOURCES.md`. Before using supervised mode, establish exact design and capture provenance; folder names are insufficient. Unknown information remains null.

The instance fallback groups exact/pixel duplicates, documented derivatives and likely near-duplicates before a seeded approximately 70/15/15 group split. One representative per group is retained. It does **not** establish unseen-design generalization. Vendor data is conservatively excluded in this fallback. Natural mode requires distinct capture groups on gallery/query sides of the same held-out designs.

Synthetic query images are deterministic RGB-channel permutations of held-out gallery originals. Secondary grayscale and RGB-complement (255-x) stress tests use the same frozen threshold. These copies are never represented as genuine colorways. Different instance groups are provisional negatives; hidden shared designs may remain.

In [2]:
from saree.data import audit, make_splits, policy_annotations
from saree.fixtures import generate
annotations = read_json(ANNOTATIONS) if ANNOTATIONS else {}
if MODE == "synthetic_smoke":
    DATA_ROOT = OUTPUT_ROOT / "fixtures"
    generate(DATA_ROOT)
    TRAINING_MODE = "instance"
    config.update(epochs=2, batch_size=4, max_seconds=120, pretrained=True, synthetic_smoke=True)
else:
    if not DATA_ROOT.is_dir() or not any(p.suffix.lower() in (".jpg", ".jpeg", ".png") for p in DATA_ROOT.rglob("*")):
        raise FileNotFoundError("Authorized real images missing. Run scripts/acquire.py or supply DATA_ROOT. No synthetic fallback.")
    policy = policy_annotations(DATA_ROOT, PROJECT_ROOT / "configs/audit_policy.json", exclude_vendor=TRAINING_MODE=="instance")
    policy.update(annotations)
    annotations = policy
write_json(OUTPUT_ROOT / "annotations.json", annotations)
manifest = OUTPUT_ROOT / "manifest.json"
split = OUTPUT_ROOT / "split.json"
audit_summary = audit(DATA_ROOT, manifest, OUTPUT_ROOT / "annotations.json")
split_summary = make_splits(manifest, split, TRAINING_MODE, seed=config["seed"])
print({k:audit_summary[k] for k in ["images","corrupt","exact_pixel_duplicates","reliable_designs"]})
print(split_summary)

{'images': 24, 'corrupt': 0, 'exact_pixel_duplicates': 0, 'reliable_designs': 0}
{'split_sha256': '9a054cc25baf134677cbefdf72db2d765b7c0744d691ff22539c1cd9db1ced44', 'counts': {'val/synthetic_gallery': 2, 'train/train': 9, 'val/excluded': 2, 'train/excluded': 3, 'test/synthetic_gallery': 2, 'test/excluded': 6}}


## Model, loss and training
MobileNetV3-Small `IMAGENET1K_V1`, adaptive mean pooling, learned 576→256 projection, L2 normalization. Frozen baselines use the actual 576-dimensional pooled features, in RGB and grayscale, with no learned head.

EXIF correction, RGB conversion, aspect-fit bilinear resize and neutral padding to 224×224 preserve the full view. ImageNet mean/std normalization follows the checkpoint; replacing its center crop is deliberate. The upstream Kaggle stretch cannot be undone. Color jitter and occasional grayscale are independent for each view; geometry is limited to small translations. Palette permutation is an evaluation stress transform.

Contrastive loss excludes self-similarity. Reliable supervised mode uses P identities × K distinct captures, then two augmented views per capture. Instance mode uses two views of each original; these are not independent observations. AdamW, one head-only warm-up epoch, then backbone fine-tuning with fixed batch-normalization statistics. The highest validation mAP selects the checkpoint, earliest on ties. CPU uses float32; optional CUDA AMP is supported.

In [3]:
from saree.training import train, baseline
checkpoint = Path(train(DATA_ROOT, split, OUTPUT_ROOT / "trained", config, device))
checkpoints = [checkpoint]
if MODE == "real":
    for kind in ("rgb", "gray"):
        ck = OUTPUT_ROOT / kind / "baseline.pt"
        baseline(kind, split, ck)
        checkpoints.append(ck)
print(read_json(OUTPUT_ROOT / "trained/training_summary.json"))

{'epoch': 1, 'loss': 1.2511807680130005, 'gradient_norm_mean': 3.456579327583313, 'validation_mAP': 1.0, 'validation_embedding_std_mean': 0.019289959222078323, 'elapsed_seconds': 1.4887941229999342, 'backbone_frozen': True, 'batches': 2}


{'epoch': 2, 'loss': 1.3252312541007996, 'gradient_norm_mean': 68.4039077758789, 'validation_mAP': 1.0, 'validation_embedding_std_mean': 0.024021143093705177, 'elapsed_seconds': 7.838305242000388, 'backbone_frozen': False, 'batches': 2}


{'best_validation_mAP': 1.0, 'epochs_executed': 2, 'real_design_claims_supported': False, 'reload_max_abs_difference': 0.0, 'seconds': 10.101820396000221, 'test_used': False, 'time_budget_reached': False}


## Validation calibration, lock, and final evaluation
Recall@K means the fraction of eligible queries with at least one relevant gallery image in the first K ranks. AP averages precision at relevant ranks over all gallery positives; mAP averages AP across eligible queries. Queries without positives are counted separately. Rankings are image-level, with stable manifest-order ties.

Verification uses the frozen pair lists and cosine scores. The threshold maximizes validation balanced accuracy, breaking ties toward the highest threshold. All model choices and thresholds are locked before a single final test invocation. ROC-AUC, precision, recall, F1 and balanced accuracy are reported with class counts. Synthetic thresholds are not proven natural same-design thresholds. Pairwise thresholds are not open-set gallery rejection thresholds.

No unsupported low-FPR or independent-pair uncertainty claim is made. Pair observations share images. Natural cross-colorway and hard different-design/similar-palette results remain unavailable unless supported by actual annotations.

In [4]:
from saree.evaluation import calibrate, lock_models, evaluate
calibrations = []
for ck in checkpoints:
    cal = ck.parent / "calibration.json"
    calibrate(ck, split, DATA_ROOT, cal, device)
    calibrations.append(cal)
lock = OUTPUT_ROOT / "experiment_lock.json"
lock_models(split, checkpoints, calibrations, lock)
results = evaluate(lock, split, DATA_ROOT, OUTPUT_ROOT / "evaluation", device)
for name, value in results.items():
    key = "recolor" if TRAINING_MODE == "instance" else "natural"
    print(name, value[key]["retrieval"], {k:v for k,v in value[key]["verification"].items() if k != "roc"})

trained_trained {'recall_at_1': 1.0, 'recall_at_5': None, 'mAP': 1.0, 'queries': 2, 'eligible_queries': 2, 'queries_without_positive': 0, 'gallery_images': 2, 'gallery_identities': 2, 'positives_per_query': [1, 1], 'ranking_unit': 'image'} {'available': True, 'roc_auc': 1.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'balanced_accuracy': 0.5, 'threshold': 0.9350371360778809, 'positive_pairs': 2, 'negative_pairs': 2, 'positive_fraction': 0.5, 'negative_count_fpr_resolution': 0.5, 'low_fpr_claim': None, 'dependence': 'pairs share queries/gallery; not independent trials'}


## Identification, pair verification and efficiency
Gallery embeddings are cached once. Cache metadata binds the checkpoint, preprocessing, frozen split and embedding-file hashes. Exact cosine matrix search returns opaque IDs and scores, not probabilities. Uploads in the optional local app stay local.

The benchmark measures encoder, preprocessing (excluding disk decode), and exact search separately with warm-up/repeats and device synchronization. Parameters, embedding storage and checkpoint size are measured. FLOPs are unavailable because no FLOP-counting tool was run.

In [5]:
from saree.retrieval import build_gallery, retrieve, verify
from saree.benchmark import benchmark
cache_dir = OUTPUT_ROOT / "gallery"
build_gallery(checkpoint, split, DATA_ROOT, cache_dir, device=device)
spec = read_json(split)
row = next(r for r in spec["rows"] if r["image_id"] == spec["protocols"]["test"]["gallery"][0])
example = DATA_ROOT / row["relative_path"]
# API sanity check on one gallery image; not an additional evaluation metric.
print(retrieve(checkpoint, cache_dir, split, example, device=device))
print(verify(checkpoint, calibrations[0], example, example, device=device))
print(benchmark(checkpoint, OUTPUT_ROOT / "efficiency.json", device=device))

[{'image_id': '6f69491247fa75ad', 'design_id': None, 'cosine_similarity': 1.0}, {'image_id': '01443d42744c5617', 'design_id': None, 'cosine_similarity': 0.7153674960136414}]


{'cosine_similarity': 0.9999999403953552, 'threshold': 0.9350371360778809, 'same_design_decision': True, 'calibration_protocol': 'synthetic_instance_recoloring', 'claim_supported': False, 'warning': 'Synthetic instance calibration only; genuine same-design decision unvalidated'}


{'checkpoint_sha256': '51550fca9a98853acbae64a5a9dbba75597cc0850089ec1aa62da65c54077b0e', 'parameters': 1074720, 'embedding_dimension': 256, 'embedding_dtype': 'float32', 'embedding_bytes': 1024, 'checkpoint_bytes': 5613391, 'checkpoint_contains_optimizer': True, 'device': 'cpu', 'hardware': '13th Gen Intel(R) Core(TM) i5-1334U', 'platform': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39', 'torch': '2.9.0+cpu', 'input_shape': [1, 3, 224, 224], 'batch_size': 1, 'precision': 'float32', 'threads': 2, 'warmup': 10, 'repeats': 50, 'accelerator_synchronized': False, 'encoder': {'median_ms': 34.84896150030181, 'p95_ms': 96.41341245001054, 'mean_ms': 44.27714550005476}, 'preprocessing_excludes_disk_decode': {'median_ms': 10.756961999959458, 'p95_ms': 15.239524150001671, 'mean_ms': 11.214304439927218}, 'preprocess_source_shape': [480, 640, 3], 'search': {'median_ms': 0.1653299996178248, 'p95_ms': 0.2960443000574741, 'mean_ms': 0.17297773993050214}, 'search_gallery_size': 1000, 's

## Limits, source disclosures and follow-up
Study `reports/EVALUATION.md`, `reports/EFFICIENCY.md`, `INTERVIEW_GUIDE.md`, and `DECISIONS.md`. Similar color can still dominate; grayscale can erase isoluminant boundaries. A 224-pixel global representation loses tiny motifs, and backgrounds/cloth folds can shortcut instance matching. Genuine annotated alternate colorways and independent captures are the next evidence needed.

Sources: [assignment dataset on Kaggle](https://www.kaggle.com/datasets/div456/indian-saree-patterns), its bundled Roboflow README, and [torchvision MobileNetV3-Small documentation](https://docs.pytorch.org/vision/0.24/models/generated/torchvision.models.mobilenet_v3_small.html). The checkpoint is `mobilenet_v3_small-047dcff4.pth` from download.pytorch.org. No external vision API processed corpus images. The vendor link is intentionally omitted from shareable code.

**AI tools used: Yes.** All visual inspection by the coding agent is AI-assisted, not human-verified. This clean notebook is an executable deliverable, not itself proof of execution. Do not embed private images in notebook outputs or upload vendor data to hosted runtimes without permission. Delete project vendor copies after the exercise. The user must verify reviewer access and submit personally by **2 October 2026, 11:59 AM IST**.